In [ ]:
import os

GIT = True
REPO_NAME = "generative_models"

Done=False
if GIT:
    if not (os.path.exists(REPO_NAME) or Done):
        !git clone https://github.com/Mayeul84/generative_models
        Done = True
import os
os.chdir(REPO_NAME)

In [ ]:
!git clone https://github.com/DPS2022/diffusion-posterior-sampling.git
!cp -r diffusion-posterior-sampling/guided_diffusion guided_diffusion
!wget -nc -O ffhq256-1k-validation.zip 'https://www.dropbox.com/scl/fi/pppstbdsf0em6o0qscruc/ffhq256-1k-validation.zip?rlkey=xl7nwv2nxb6yvsirr3wad77hm'
!unzip -nq ffhq256-1k-validation.zip
!wget -nc -O ffhq_10m.pt 'https://www.dropbox.com/scl/fi/pq72vxzxcbygieq5z4gvf/ffhq_10m.pt?rlkey=5sxdj6r4o9f7b7bbp5fxg2f5r'

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp
from scipy.sparse.linalg import splu
from PIL import Image
from tqdm import tqdm
from guided_diffusion.unet import create_model

from tqdm.notebook import tqdm, trange

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", device)

import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

Device: cpu


In [ ]:
from utils import *

idx = np.random.randint(1000)
print('Image', str(idx).zfill(5))
img_pil = Image.open('ffhq256-1k-validation/'+str(idx).zfill(5)+'.png')
display(img_pil)
display_as_pilimg(pilimg_to_tensor(img_pil));
img = pilimg_to_tensor(img_pil)

Image 00291


FileNotFoundError: [Errno 2] No such file or directory: 'ffhq256-1k-validation/00291.png'

In [ ]:
# Load model
model_config = {'image_size': 256,
                'num_channels': 128,
                'num_res_blocks': 1,
                'channel_mult': '',
                'learn_sigma': True,
                'class_cond': False,
                'use_checkpoint': False,
                'attention_resolutions': 16,
                'num_heads': 4,
                'num_head_channels': 64,
                'num_heads_upsample': -1,
                'use_scale_shift_norm': True,
                'dropout': 0.0,
                'resblock_updown': True,
                'use_fp16': False,
                'use_new_attention_order': False,
                'model_path': 'ffhq_10m.pt'}


model = create_model(**model_config)
model = model.to(device)
# use in eval mode:
model.eval()

In [ ]:
from df_models import DDPM
ddpm = DDPM(num_diffusion_timesteps=1000, beta_start=0.0001, beta_end=0.002, imgshape=img.shape, model=model)

### Inpainting

In [ ]:
from utils_operator import Inpainting
from utils import *

operator = Inpainting(mask=0.5,imgshape=(256,256), device=device)

idx = 12
x_true_pil = Image.open('ffhq256-1k-validation/'+str(idx).zfill(5)+'.png')
x_true = pilimg_to_tensor(x_true_pil)[:,:3,:,:]
print(x_true.device)
print("original image", str(idx).zfill(5)+'.png')
display_as_pilimg(x_true)

y = operator.linear_operator(x_true.clone())
print("Inpainting image")
display_as_pilimg(y);

### PnP

In [ ]:
from algo import PNP_SGS

pnp_config = {
    "ro":0.1,
    "MCMC_steps":110,
    "x_true":x_true,
    "y":y,
    "Burn_in_steps":20,
    "diffusing_model":model,
    "operator":operator,
    "show_only_last":True
}

x_samples, time = PNP_SGS(**pnp_config) 